# 00 Gold Data Profiling

## Purpose

The Silver layer is finished. Before any Gold table is built, this notebook measures the facts that the Gold rules depend on. It is **step 2** of the Gold layer plan.

**This notebook only reads tables.** It creates nothing and changes nothing, so it is safe to run more than once.

## The twelve checks

| # | Check | Why it matters |
|---|---|---|
| 1 | Files per source date in Bronze, and rows per file | Input for the source-hour manifest that G5 needs |
| 2 | Each `quarantine_warning` value in the nine Silver tables | Shows how many rows a metric will leave out |
| 3 | Repository-days per Silver table and per daily Gold table | Gives the expected row count of each daily Gold table |
| 4 | Pull requests: identity, closed events, close time | Confirms the merge share and duration rules of G1 |
| 5 | Issues: the same questions | Confirms the resolution time rule of G2 |
| 6 | Releases: draft and prerelease flags | Needed for the prerelease rule of G5 |
| 7 | Releases: several events for one release | Confirms the rule "one anchor per release" |
| 8 | Releases: published time and the data window | Shows how many releases can become a G5 row |
| 9 | Releases per repository and the activity around them | Checks that the G5 join stays small enough |
| 10 | Repositories with more than one name | Confirms the rule for the `repo_name` label |
| 11 | Bots and the most active actors | Shows how much bots matter in G3 |
| 12 | Is `event_id` unique across all nine Silver tables? | G3 and G6 stack tables and must not count an event twice |

There is also a short starting check (**R0**) that confirms the Silver tables are the ones we expect.

## How to read the results

- Every result table has a label such as **R4.2**. The label is the first comment line of its cell.
- "Expected" numbers come from the Silver profiling and from the audit rows of the all-dates Job run.
- A different number is **not** an error of this notebook. It is a finding, and we look at it together before the Gold contract is frozen.

## How to run it

1. Attach **serverless notebook compute** (not a SQL warehouse, because the cells are Python).
2. Click **Run all**.
3. Send the notebook back with its results: commit it with its outputs to this branch, or export it as `.ipynb` and upload it.

## Setup

This cell sets the table names, the data window and three small helper functions that the checks below reuse:

- `count_when(condition)` counts the rows where a condition is true. A `NULL` condition is not counted.
- `union_all(parts)` places DataFrames with the same columns under each other.
- `stack(names)` places the chosen Silver tables under each other and keeps only the eight common columns. A label column `silver_table` says where each row came from.

The four lists at the end name the Silver tables that each Gold table will read.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# All timestamps in this project are UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")


BRONZE_TABLE = "github_lakehouse.bronze.github_events_raw"

# Short name -> Silver table. The short names are used as labels in the results.
SILVER_TABLES = {
    "push": "github_lakehouse.silver.push_events",
    "pull_request": "github_lakehouse.silver.pull_request_events",
    "issue": "github_lakehouse.silver.issue_events",
    "issue_comment": "github_lakehouse.silver.issue_comment_events",
    "review": "github_lakehouse.silver.pull_request_review_events",
    "review_comment": "github_lakehouse.silver.pull_request_review_comment_events",
    "watch": "github_lakehouse.silver.watch_events",
    "fork": "github_lakehouse.silver.fork_events",
    "release": "github_lakehouse.silver.release_events",
}

# Silver row counts written by the all-dates Job run (audit table).
EXPECTED_SILVER_ROWS = {
    "push": 26536967,
    "pull_request": 2173958,
    "issue": 548956,
    "issue_comment": 1357324,
    "review": 717277,
    "review_comment": 422917,
    "watch": 1380559,
    "fork": 326417,
    "release": 159416,
}

# The data window of this project (source dates, UTC).
FIRST_SOURCE_DATE = "2024-02-01"
LAST_SOURCE_DATE = "2024-02-07"

# First moment inside the data window, and first moment after it.
DATA_START = F.to_timestamp(F.lit(FIRST_SOURCE_DATE + " 00:00:00"))
DATA_END = F.to_timestamp(F.lit(LAST_SOURCE_DATE + " 00:00:00")) + F.expr("INTERVAL 1 DAY")

# The eight columns that every Silver table has.
COMMON_COLUMNS = [
    "event_id",
    "event_type",
    "event_timestamp",
    "event_date",
    "actor_id",
    "actor_login",
    "repo_id",
    "repo_name",
]


bronze_df = spark.table(BRONZE_TABLE)

silver = {
    name: spark.table(table_name)
    for name, table_name in SILVER_TABLES.items()
}


def count_when(condition):
    """Count the rows where the condition is true. NULL is not counted."""
    return F.sum(F.when(condition, 1).otherwise(0))


def union_all(parts):
    """Place DataFrames that have the same columns under each other."""
    result_df = parts[0]

    for part_df in parts[1:]:
        result_df = result_df.unionByName(part_df)

    return result_df


def stack(names):
    """Place the chosen Silver tables under each other (common columns only)."""
    parts = []

    for name in names:
        parts.append(
            silver[name].select(
                F.lit(name).alias("silver_table"),
                *COMMON_COLUMNS
            )
        )

    return union_all(parts)


# The Silver tables each Gold table will read.
G3_SOURCES = ["push", "pull_request", "issue", "issue_comment", "review", "review_comment"]
G4_SOURCES = ["issue_comment", "review", "review_comment"]
G5_ACTIVITY_SOURCES = ["push", "pull_request", "issue", "watch", "fork"]
G6_SOURCES = list(SILVER_TABLES)

## Starting check: are the Silver tables the ones we expect?

**R0.1** shows one row per Silver table.

What we expect:

- `rows` equals `expected_rows` for every table, so the last column `rows_minus_expected` is 0. The nine tables hold 33,623,791 rows in total.
- The first event is on 2024-02-01 and the last one on 2024-02-07.
- All seven `null_...` columns are 0. Gold leaves out rows without an event date or a repository, and it counts rows without an actor separately.
- `event_date_differs_from_source_date` is 0. The refresh design uses the source date to find the days a run has to rebuild.

In [0]:
# R0.1 Silver baseline: one row per Silver table

baseline_parts = []

for name, silver_df in silver.items():
    baseline_parts.append(
        silver_df
        .agg(
            F.count("*").alias("rows"),
            F.min("event_timestamp").alias("first_event_time"),
            F.max("event_timestamp").alias("last_event_time"),
            count_when(F.col("event_id").isNull()).alias("null_event_id"),
            count_when(F.col("event_timestamp").isNull()).alias("null_event_timestamp"),
            count_when(F.col("event_date").isNull()).alias("null_event_date"),
            count_when(F.col("repo_id").isNull()).alias("null_repo_id"),
            count_when(F.col("repo_name").isNull()).alias("null_repo_name"),
            count_when(F.col("actor_id").isNull()).alias("null_actor_id"),
            count_when(F.col("actor_login").isNull()).alias("null_actor_login"),
            count_when(
                F.col("event_date") != F.col("_source_date")
            ).alias("event_date_differs_from_source_date"),
        )
        .select(
            F.lit(name).alias("silver_table"),
            F.lit(EXPECTED_SILVER_ROWS[name]).alias("expected_rows"),
            "*",
        )
        .withColumn(
            "rows_minus_expected",
            F.col("rows") - F.col("expected_rows"),
        )
    )

display(union_all(baseline_parts))

## 1. Bronze: files per source date and rows per file

GH Archive delivers one file per hour, for example `2024-02-01-0.json.gz` for the first hour of 1 February. G5 compares the 24 hours before a release with the 24 hours after it, so it needs proof that no hour is missing. This check is the input for the source-hour manifest.

Only the lineage columns of Bronze are read. The raw JSON is not touched.

- **R1.1** Files and rows per source date. Expected: 24 files for each of the 7 dates, 38,555,222 rows in total.
- **R1.2** Every expected hour compared with the files found. Expected: one line, `OK` with 168 hours.
- **R1.3** The hours that are not `OK`. Expected: no rows.
- **R1.4** All files with their row counts (168 rows). We keep this list to compare with the manifest later.

The findings in R1.2 mean:

| Finding | Meaning |
|---|---|
| `OK` | The expected file was found once, under the right source date |
| `EXPECTED_FILE_MISSING` | No Bronze row comes from the file of this hour |
| `UNEXPECTED_FILE` | Bronze has a file name that is not one of the 168 expected names |
| `FILE_UNDER_WRONG_SOURCE_DATE` | The date in the file name differs from `_source_date` |
| `MORE_THAN_ONE_INGESTION_TIME` | The file was loaded at more than one time, so its rows may be doubled |

In [0]:
# R1.1 Files and rows per source date

# One row per source file.
bronze_files_df = (
    bronze_df
    .groupBy("_source_date", "_source_file_name")
    .agg(
        F.count("*").alias("bronze_rows"),
        F.countDistinct("_ingested_at").alias("ingestion_times"),
        F.min("_ingested_at").alias("first_ingested_at"),
    )
)

display(
    bronze_files_df
    .groupBy("_source_date")
    .agg(
        F.count("*").alias("source_files"),
        F.sum("bronze_rows").alias("bronze_rows"),
        F.min("bronze_rows").alias("smallest_file_rows"),
        F.max("bronze_rows").alias("largest_file_rows"),
    )
    .orderBy("_source_date")
)

In [0]:
# R1.2 Every expected hour compared with the files found

# One row per hour of the data window, with the file name GH Archive uses for it.
expected_hours_df = (
    spark.sql(f"""
        SELECT explode(sequence(
            TIMESTAMP '{FIRST_SOURCE_DATE} 00:00:00',
            TIMESTAMP '{LAST_SOURCE_DATE} 23:00:00',
            INTERVAL 1 HOUR
        )) AS source_hour_utc
    """)
    .withColumn(
        "expected_file_name",
        F.concat(
            F.date_format("source_hour_utc", "yyyy-MM-dd"),
            F.lit("-"),
            F.hour("source_hour_utc").cast("string"),
            F.lit(".json.gz"),
        ),
    )
)

# A full outer join keeps both sides: expected hours without a file,
# and files that are not expected.
hour_check_df = (
    expected_hours_df
    .join(
        bronze_files_df,
        F.col("expected_file_name") == F.col("_source_file_name"),
        "full_outer",
    )
    .withColumn(
        "finding",
        F.when(F.col("_source_file_name").isNull(), "EXPECTED_FILE_MISSING")
        .when(F.col("expected_file_name").isNull(), "UNEXPECTED_FILE")
        .when(
            F.col("_source_date") != F.to_date("source_hour_utc"),
            "FILE_UNDER_WRONG_SOURCE_DATE",
        )
        .when(F.col("ingestion_times") > 1, "MORE_THAN_ONE_INGESTION_TIME")
        .otherwise("OK"),
    )
)

display(
    hour_check_df
    .groupBy("finding")
    .agg(
        F.count("*").alias("hours_or_files"),
        F.sum("bronze_rows").alias("bronze_rows"),
    )
    .orderBy("finding")
)

In [0]:
# R1.3 The hours that are not OK (expected: no rows)

display(
    hour_check_df
    .filter(F.col("finding") != "OK")
    .select(
        "finding",
        "source_hour_utc",
        "expected_file_name",
        "_source_file_name",
        "_source_date",
        "bronze_rows",
        "ingestion_times",
    )
    .orderBy("source_hour_utc", "_source_file_name")
)

In [0]:
# R1.4 All files with their row counts (168 rows expected)

display(
    hour_check_df
    .select(
        "source_hour_utc",
        "_source_date",
        "_source_file_name",
        "bronze_rows",
        "first_ingested_at",
        "finding",
    )
    .orderBy("source_hour_utc", "_source_file_name")
)

## 2. Warnings in the nine Silver tables

Silver keeps a row with a non-fatal problem and writes the reason into `quarantine_warning`, for example `MISSING: pr_closed_at`. A row with several problems has them in one text, separated by `; `. A clean row has `NULL`.

In Gold, such a row still counts as an event, but a metric that needs the missing value leaves it out. This check shows how many rows that will be.

- **R2.1** Rows per table and full warning text. `null` is the clean rows.
- **R2.2** Rows per table and single warning. A row with two warnings is counted under both.

The only warning we know so far is `MISSING: pr_closed_at` with 148 rows in `pull_request`.

In [0]:
# R2.1 Rows per Silver table and full warning text

warning_parts = []

for name, silver_df in silver.items():
    warning_parts.append(
        silver_df
        .groupBy("quarantine_warning")
        .agg(F.count("*").alias("rows"))
        .select(
            F.lit(name).alias("silver_table"),
            "quarantine_warning",
            "rows",
        )
    )

warnings_df = union_all(warning_parts)

display(
    warnings_df
    .orderBy("silver_table", F.desc("rows"))
)

In [0]:
# R2.2 Rows per Silver table and single warning

display(
    warnings_df
    .filter(F.col("quarantine_warning").isNotNull())
    .withColumn(
        "single_warning",
        F.explode(F.split("quarantine_warning", "; ")),
    )
    .groupBy("silver_table", "single_warning")
    .agg(F.sum("rows").alias("rows"))
    .orderBy("silver_table", F.desc("rows"))
)

## 3. Repository-days

Five Gold tables have one row per repository and UTC day. This check counts how many such rows there will be, and how much data stands behind a typical row.

- **R3.1** One row per Silver table: events, repository-days and repositories.
- **R3.2** One row per daily Gold table. `repository_days` is the expected row count of that Gold table for the whole week. The `days_with_...` columns show how many of those rows rest on very few events. A percentage on one or two events is correct, but it says little.
- **R3.3** The same per day (35 rows). We use it later to check a Job run for one date.

G5 is not in this check, because it has one row per release (checks 6 to 9).

What we expect: the `events` column of R3.2 is 2,173,958 for G1, 548,956 for G2, 31,757,399 for G3, 2,497,518 for G4 and 33,623,791 for G6.

In [0]:
# R3.1 Repository-days per Silver table

silver_grain_parts = []

for name, silver_df in silver.items():
    silver_grain_parts.append(
        silver_df
        .agg(
            F.count("*").alias("events"),
            F.countDistinct("repo_id", "event_date").alias("repository_days"),
            F.countDistinct("repo_id").alias("repositories"),
        )
        .select(F.lit(name).alias("silver_table"), "*")
    )

display(union_all(silver_grain_parts))

In [0]:
# R3.2 Repository-days per daily Gold table

GOLD_DAILY_SOURCES = {
    "G1 repository_delivery_metrics": ["pull_request"],
    "G2 repository_issue_flow": ["issue"],
    "G3 repository_contributor_concentration": G3_SOURCES,
    "G4 repository_collaboration_metrics": G4_SOURCES,
    "G6 repository_activity_profile": G6_SOURCES,
}

gold_grain_parts = []
gold_per_date_parts = []

for gold_table, source_names in GOLD_DAILY_SOURCES.items():

    # One row per repository and day, with its number of events.
    per_day_df = (
        stack(source_names)
        .groupBy("repo_id", "event_date")
        .agg(F.count("*").alias("events"))
    )

    gold_grain_parts.append(
        per_day_df
        .agg(
            F.sum("events").alias("events"),
            F.count("*").alias("repository_days"),
            F.countDistinct("repo_id").alias("repositories"),
            count_when(F.col("events") == 1).alias("days_with_1_event"),
            count_when(F.col("events") == 2).alias("days_with_2_events"),
            count_when(F.col("events").between(3, 9)).alias("days_with_3_to_9_events"),
            count_when(F.col("events") >= 10).alias("days_with_10_or_more_events"),
            F.max("events").alias("most_events_on_one_day"),
        )
        .select(F.lit(gold_table).alias("gold_table"), "*")
    )

    gold_per_date_parts.append(
        per_day_df
        .groupBy("event_date")
        .agg(
            F.count("*").alias("repository_days"),
            F.sum("events").alias("events"),
        )
        .select(F.lit(gold_table).alias("gold_table"), "*")
    )

display(
    union_all(gold_grain_parts)
    .withColumn(
        "pct_days_with_1_or_2_events",
        F.round(
            F.try_divide(
                100 * (F.col("days_with_1_event") + F.col("days_with_2_events")),
                F.col("repository_days"),
            ),
            2,
        ),
    )
    .orderBy("gold_table")
)

In [0]:
# R3.3 Repository-days per daily Gold table and day (35 rows expected)

display(
    union_all(gold_per_date_parts)
    .orderBy("gold_table", "event_date")
)

## 4. Pull requests (G1)

G1 counts pull request events and measures how long merged and closed pull requests took.

**How a pull request is identified:** by `pr_id`. If it is missing, by `repo_id` plus `pr_number`. If both are missing, the row still counts as an event, but not as a pull request.

- **R4.1** Identity: how many rows have no usable key, and how many different pull requests there are.
- **R4.2** Closed events per pull request and day. The line with `1` is the normal case. `4` means four or more. Lines above `1` are pull requests that were closed more than once on the same day. Only for those does "share of closed events" differ from "share of closed pull requests".
- **R4.3** The closed events without a close time, grouped. Expected: 148 rows in total.
- **R4.4** A sample of 20 of those rows, so we can see what they look like.
- **R4.5** Per action: how many rows carry a close time or a merge time. Profiling found more rows with a close time (1,037,160) than `closed` actions (1,031,732), so other actions carry a close time too. That is why G1 durations require the `closed` action.
- **R4.6** A preview of the G1 rules on the whole week. Nothing is written.
- **R4.7** How long merging and closing took, in hours, on the eligible rows. The median and the other percentiles are close estimates here, which is enough for a first look.

The columns of R4.6:

| Column | Meaning |
|---|---|
| `closed_actions` | Rows with the action `closed` |
| `eligible_closed_rows` | Of those, rows where the merged flag is known |
| `merged_closed_rows` | Of those, rows where the pull request was merged |
| `merge_share_pct` | 100 × merged ÷ eligible |
| `merged_duration_eligible_rows` | Merged rows with a creation time and a merge time that is not earlier |
| `closed_duration_eligible_rows` | Closed rows with a creation time and a close time that is not earlier |
| `merged_without_merge_time` | Closed and merged, but no merge time. Left out of the merge duration |
| `merged_on_other_actions` | Merged flag true on a row that is not `closed`. Not used by G1 |

In [0]:
# R4.1 Pull request identity

pr_df = silver["pull_request"]

# pr_id first. Without it, repository plus number. NULL when both are missing.
# concat() returns NULL as soon as one part is NULL, which is what we want here.
pr_key = F.coalesce(
    F.concat(F.lit("id:"), F.col("pr_id").cast("string")),
    F.concat(
        F.lit("number:"),
        F.col("repo_id").cast("string"),
        F.lit("#"),
        F.col("pr_number").cast("string"),
    ),
)

display(
    pr_df
    .agg(
        F.count("*").alias("pr_events"),
        count_when(F.col("pr_id").isNull()).alias("null_pr_id"),
        count_when(F.col("pr_number").isNull()).alias("null_pr_number"),
        count_when(pr_key.isNull()).alias("no_usable_pr_key"),
        F.countDistinct("pr_id").alias("distinct_pr_id"),
        F.countDistinct("repo_id", "pr_number").alias("distinct_repo_and_number"),
        F.countDistinct(pr_key).alias("distinct_pull_requests"),
    )
)

In [0]:
# R4.2 Closed events per pull request and day (4 = four or more)

closed_per_pr_day_df = (
    pr_df
    .filter(F.col("pr_action") == "closed")
    .withColumn("pr_key", pr_key)
    .filter(F.col("pr_key").isNotNull())
    .groupBy("pr_key", "event_date")
    .agg(F.count("*").alias("closed_events"))
)

display(
    closed_per_pr_day_df
    .withColumn(
        "closed_events_that_day",
        F.least(F.col("closed_events"), F.lit(4)),
    )
    .groupBy("closed_events_that_day")
    .agg(
        F.count("*").alias("pull_request_days"),
        F.sum("closed_events").alias("closed_events"),
    )
    .orderBy("closed_events_that_day")
)

In [0]:
# R4.3 Closed events without a close time, grouped (148 rows expected in total)

pr_closed_without_time_df = pr_df.filter(
    (F.col("pr_action") == "closed")
    & F.col("pr_closed_at").isNull()
)

display(
    pr_closed_without_time_df
    .groupBy(
        "pr_merged",
        F.col("pr_merged_at").isNotNull().alias("has_merge_time"),
        F.col("pr_created_at").isNotNull().alias("has_creation_time"),
        "quarantine_warning",
    )
    .agg(F.count("*").alias("rows"))
    .orderBy(F.desc("rows"))
)

In [0]:
# R4.4 A sample of 20 closed events without a close time

display(
    pr_closed_without_time_df
    .select(
        "event_id",
        "event_timestamp",
        "repo_name",
        "pr_number",
        "pr_id",
        "pr_created_at",
        "pr_closed_at",
        "pr_merged_at",
        "pr_merged",
        "quarantine_warning",
    )
    .orderBy("event_timestamp", "event_id")
    .limit(20)
)

In [0]:
# R4.5 Close time and merge time per action

display(
    pr_df
    .groupBy("pr_action")
    .agg(
        F.count("*").alias("rows"),
        count_when(F.col("pr_closed_at").isNotNull()).alias("with_close_time"),
        count_when(F.col("pr_merged_at").isNotNull()).alias("with_merge_time"),
        count_when(F.col("pr_merged")).alias("merged_true"),
        count_when(~F.col("pr_merged")).alias("merged_false"),
        count_when(F.col("pr_merged").isNull()).alias("merged_unknown"),
    )
    .orderBy(F.desc("rows"))
)

In [0]:
# R4.6 Preview of the G1 rules on the whole week

pr_is_closed = F.col("pr_action") == "closed"
pr_is_merged = F.col("pr_merged")

# Hours from creation to merge, and from creation to close.
# The result is NULL when one of the two times is missing.
pr_merge_hours = (
    F.unix_timestamp("pr_merged_at") - F.unix_timestamp("pr_created_at")
) / 3600

pr_close_hours = (
    F.unix_timestamp("pr_closed_at") - F.unix_timestamp("pr_created_at")
) / 3600

merge_duration_ok = pr_is_closed & pr_is_merged & (pr_merge_hours >= 0)
close_duration_ok = pr_is_closed & (pr_close_hours >= 0)

display(
    pr_df
    .agg(
        count_when(pr_is_closed).alias("closed_actions"),
        count_when(
            pr_is_closed & F.col("pr_merged").isNotNull()
        ).alias("eligible_closed_rows"),
        count_when(pr_is_closed & pr_is_merged).alias("merged_closed_rows"),
        count_when(merge_duration_ok).alias("merged_duration_eligible_rows"),
        count_when(close_duration_ok).alias("closed_duration_eligible_rows"),
        count_when(
            pr_is_closed & pr_is_merged & F.col("pr_merged_at").isNull()
        ).alias("merged_without_merge_time"),
        count_when(~pr_is_closed & pr_is_merged).alias("merged_on_other_actions"),
    )
    .withColumn(
        "merge_share_pct",
        F.round(
            F.try_divide(
                100 * F.col("merged_closed_rows"),
                F.col("eligible_closed_rows"),
            ),
            2,
        ),
    )
)

In [0]:
# R4.7 Hours to merge and hours to close, on the eligible rows

def duration_profile(df, label, hours):
    """One row that describes a duration: how many rows, and typical values."""
    return (
        df
        .select(hours.alias("hours"))
        .agg(
            F.count("*").alias("eligible_rows"),
            F.round(F.min("hours"), 2).alias("shortest_hours"),
            F.round(F.percentile_approx("hours", 0.5), 2).alias("median_hours"),
            F.round(F.percentile_approx("hours", 0.9), 2).alias("p90_hours"),
            F.round(F.percentile_approx("hours", 0.99), 2).alias("p99_hours"),
            F.round(F.max("hours"), 2).alias("longest_hours"),
            F.round(F.avg("hours"), 2).alias("average_hours"),
            count_when(F.col("hours") > 7 * 24).alias("longer_than_7_days"),
        )
        .select(F.lit(label).alias("duration"), "*")
    )


display(
    union_all([
        duration_profile(
            pr_df.filter(merge_duration_ok),
            "pull request: creation to merge",
            pr_merge_hours,
        ),
        duration_profile(
            pr_df.filter(close_duration_ok),
            "pull request: creation to close",
            pr_close_hours,
        ),
    ])
)

## 5. Issues (G2)

The same questions as for pull requests. G2 measures the time from the creation of an issue to a `closed` action.

**How an issue is identified:** by `issue_id`. If it is missing, by `repo_id` plus `issue_number`.

- **R5.1** Identity: rows without a usable key, and the number of different issues.
- **R5.2** Closed events per issue and day. `4` means four or more.
- **R5.3** Closed events without a close time, grouped. Profiling never measured this number.
- **R5.4** A sample of up to 20 of those rows.
- **R5.5** Per action: how many rows carry a close time. Profiling found 218,840 rows with a close time and 217,901 `closed` actions.
- **R5.6** A preview of the G2 rule on the whole week.
- **R5.7** Hours from creation to close on the eligible rows.

Silver already sets a close time to `NULL` when it is earlier than the creation time, and writes a warning. Such rows appear in R5.3.

In [0]:
# R5.1 Issue identity

issue_df = silver["issue"]

# issue_id first. Without it, repository plus number. NULL when both are missing.
issue_key = F.coalesce(
    F.concat(F.lit("id:"), F.col("issue_id").cast("string")),
    F.concat(
        F.lit("number:"),
        F.col("repo_id").cast("string"),
        F.lit("#"),
        F.col("issue_number").cast("string"),
    ),
)

display(
    issue_df
    .agg(
        F.count("*").alias("issue_events"),
        count_when(F.col("issue_id").isNull()).alias("null_issue_id"),
        count_when(F.col("issue_number").isNull()).alias("null_issue_number"),
        count_when(issue_key.isNull()).alias("no_usable_issue_key"),
        F.countDistinct("issue_id").alias("distinct_issue_id"),
        F.countDistinct("repo_id", "issue_number").alias("distinct_repo_and_number"),
        F.countDistinct(issue_key).alias("distinct_issues"),
    )
)

In [0]:
# R5.2 Closed events per issue and day (4 = four or more)

closed_per_issue_day_df = (
    issue_df
    .filter(F.col("issue_action") == "closed")
    .withColumn("issue_key", issue_key)
    .filter(F.col("issue_key").isNotNull())
    .groupBy("issue_key", "event_date")
    .agg(F.count("*").alias("closed_events"))
)

display(
    closed_per_issue_day_df
    .withColumn(
        "closed_events_that_day",
        F.least(F.col("closed_events"), F.lit(4)),
    )
    .groupBy("closed_events_that_day")
    .agg(
        F.count("*").alias("issue_days"),
        F.sum("closed_events").alias("closed_events"),
    )
    .orderBy("closed_events_that_day")
)

In [0]:
# R5.3 Closed events without a close time, grouped

issue_closed_without_time_df = issue_df.filter(
    (F.col("issue_action") == "closed")
    & F.col("issue_closed_at").isNull()
)

display(
    issue_closed_without_time_df
    .groupBy(
        F.col("issue_created_at").isNotNull().alias("has_creation_time"),
        "quarantine_warning",
    )
    .agg(F.count("*").alias("rows"))
    .orderBy(F.desc("rows"))
)

In [0]:
# R5.4 A sample of up to 20 closed events without a close time

display(
    issue_closed_without_time_df
    .select(
        "event_id",
        "event_timestamp",
        "repo_name",
        "issue_number",
        "issue_id",
        "issue_created_at",
        "issue_closed_at",
        "quarantine_warning",
    )
    .orderBy("event_timestamp", "event_id")
    .limit(20)
)

In [0]:
# R5.5 Close time per action

display(
    issue_df
    .groupBy("issue_action")
    .agg(
        F.count("*").alias("rows"),
        count_when(F.col("issue_closed_at").isNotNull()).alias("with_close_time"),
        count_when(F.col("issue_created_at").isNull()).alias("without_creation_time"),
    )
    .orderBy(F.desc("rows"))
)

In [0]:
# R5.6 Preview of the G2 rule on the whole week

issue_is_closed = F.col("issue_action") == "closed"

# Hours from creation to close. NULL when one of the two times is missing.
issue_resolution_hours = (
    F.unix_timestamp("issue_closed_at") - F.unix_timestamp("issue_created_at")
) / 3600

resolution_ok = issue_is_closed & (issue_resolution_hours >= 0)

display(
    issue_df
    .agg(
        count_when(issue_is_closed).alias("closed_actions"),
        count_when(resolution_ok).alias("issue_close_duration_eligible_rows"),
        count_when(
            issue_is_closed & F.col("issue_closed_at").isNull()
        ).alias("closed_without_close_time"),
        count_when(
            issue_is_closed & F.col("issue_created_at").isNull()
        ).alias("closed_without_creation_time"),
        count_when(
            ~issue_is_closed & F.col("issue_closed_at").isNotNull()
        ).alias("close_time_on_other_actions"),
    )
)

In [0]:
# R5.7 Hours from creation to close, on the eligible rows

# duration_profile() is the helper from the cell of R4.7.
display(
    duration_profile(
        issue_df.filter(resolution_ok),
        "issue: creation to close",
        issue_resolution_hours,
    )
)

## 6. Releases: draft and prerelease flags (G5)

A release can become a G5 row only when it is published, it is not a draft, and we know whether it is a prerelease. Profiling counted only the `NULL` values of the two flags (0 for both). It never counted how many are true and how many are false.

- **R6.1** Rows per action, draft flag and prerelease flag. Expected: 159,416 rows in total, all with the action `published`.

An unknown flag is never treated as false. Prereleases stay in G5 and get the column `is_prerelease`.

In [0]:
# R6.1 Release events per action, draft flag and prerelease flag

release_df = silver["release"]

display(
    release_df
    .groupBy("release_action", "release_draft", "release_prerelease")
    .agg(F.count("*").alias("rows"))
    .orderBy(F.desc("rows"))
)

## 7. Releases: several events for one release (G5)

G5 has one row per release, not one row per release event. A release is identified by `repo_id` plus `release_id`. When a release has several events, the earliest one becomes the anchor and the others are counted as duplicates.

- **R7.1** Identity: missing IDs and the number of different releases. `distinct_repo_and_tag` is close to the number of releases when every release has its own tag.
- **R7.2** Events per release. The line with `1` is the normal case. `4` means four or more. The last column counts releases whose events do not all carry the same published time.
- **R7.3** The 20 releases with the most events, so we can see what repeated events look like.

In [0]:
# R7.1 Release identity

display(
    release_df
    .agg(
        F.count("*").alias("release_events"),
        count_when(F.col("release_id").isNull()).alias("null_release_id"),
        count_when(F.col("release_tag").isNull()).alias("null_release_tag"),
        F.countDistinct("repo_id", "release_id").alias("distinct_releases"),
        F.countDistinct("release_id").alias("distinct_release_id"),
        F.countDistinct("repo_id", "release_tag").alias("distinct_repo_and_tag"),
        F.countDistinct("repo_id").alias("repositories"),
    )
)

In [0]:
# R7.2 Events per release (4 = four or more)

# One row per release.
per_release_df = (
    release_df
    .filter(F.col("release_id").isNotNull())
    .groupBy("repo_id", "release_id")
    .agg(
        F.count("*").alias("events"),
        F.countDistinct("release_published_at").alias("published_times"),
        F.min("release_published_at").alias("first_published_at"),
        F.max("release_published_at").alias("last_published_at"),
        F.min("event_timestamp").alias("first_event_time"),
        F.max("event_timestamp").alias("last_event_time"),
        F.max("repo_name").alias("repo_name"),
        F.max("release_tag").alias("release_tag"),
    )
)

display(
    per_release_df
    .withColumn(
        "events_per_release",
        F.least(F.col("events"), F.lit(4)),
    )
    .groupBy("events_per_release")
    .agg(
        F.count("*").alias("releases"),
        F.sum("events").alias("release_events"),
        count_when(
            F.col("published_times") > 1
        ).alias("releases_with_more_than_one_published_time"),
    )
    .orderBy("events_per_release")
)

In [0]:
# R7.3 The 20 releases with the most events

display(
    per_release_df
    .filter(F.col("events") > 1)
    .select(
        "repo_name",
        "release_tag",
        "release_id",
        "events",
        "published_times",
        "first_published_at",
        "last_published_at",
        "first_event_time",
        "last_event_time",
    )
    .orderBy(F.desc("events"), "repo_name", "release_id")
    .limit(20)
)

## 8. Releases: published time and the data window (G5)

G5 uses `release_published_at` as the release time. It compares the 24 hours before that time with the 24 hours after it. Both windows must lie completely inside our data, which runs from 2024-02-01 00:00 up to, but not including, 2024-02-08 00:00 UTC. So a release fits only when it was published from 2024-02-02 00:00 to 2024-02-07 00:00.

- **R8.1** The gap between the published time and the event time. A release that was published long before its event arrived may lie outside the data window.
- **R8.2** Where the published time lies compared with the data window. Only the line `3 both windows fit` can become a G5 row.
- **R8.3** A preview of the seven anchor rules. Each release event is counted once, under the first rule it fails. `ANCHOR` is the expected number of G5 rows, and `of_those_prereleases` shows how many of them carry `is_prerelease = true`.

The seven rules, in this order:

| # | A release event becomes an anchor only if | Otherwise it is counted as |
|---|---|---|
| 1 | `release_action` is `published` | `NOT_PUBLISHED` |
| 2 | `release_published_at` is known | `MISSING_PUBLISHED_AT` (278 expected) |
| 3 | `release_draft` is known and false | `DRAFT_OR_UNKNOWN` |
| 4 | `release_prerelease` is known | `PRERELEASE_UNKNOWN` |
| 5 | `release_id` is known | `MISSING_RELEASE_ID` |
| 6 | it is the earliest event of its release, among the events that passed rules 1 to 5 | `DUPLICATE_RELEASE_EVENT` |
| 7 | both 24-hour windows lie inside the data window | `WINDOW_NOT_COVERED` |

Rule 7 is checked here by time only. The real G5 notebook also asks the source-hour manifest whether every hour of the two windows was loaded.

Profiling did a simpler check on Bronze: 108,876 published releases had 24 hours on both sides, 50,262 did not, and 278 had no published time. `ANCHOR` should be close to 108,876 and somewhat lower, because drafts and repeated events are removed here.

In [0]:
# R8.1 Gap between the published time and the event time

published_at = F.col("release_published_at")

gap_minutes = (
    F.unix_timestamp("event_timestamp") - F.unix_timestamp(published_at)
) / 60

display(
    release_df
    .withColumn(
        "event_arrived",
        F.when(
            published_at.isNull() | F.col("event_timestamp").isNull(),
            "0 a time is missing",
        )
        .when(gap_minutes < 0, "1 before the published time")
        .when(gap_minutes <= 1, "2 within 1 minute after it")
        .when(gap_minutes <= 60, "3 within 1 hour after it")
        .when(gap_minutes <= 24 * 60, "4 within 24 hours after it")
        .when(gap_minutes <= 7 * 24 * 60, "5 within 7 days after it")
        .otherwise("6 more than 7 days after it"),
    )
    .groupBy("event_arrived")
    .agg(F.count("*").alias("release_events"))
    .orderBy("event_arrived")
)

In [0]:
# R8.2 Where the published time lies compared with the data window

ONE_DAY = F.expr("INTERVAL 24 HOURS")

window_position = (
    F.when(published_at.isNull(), "0 no published time")
    .when(published_at < DATA_START, "1 before the data window")
    .when(published_at < DATA_START + ONE_DAY, "2 first 24 hours: before window not complete")
    .when(published_at <= DATA_END - ONE_DAY, "3 both windows fit")
    .when(published_at < DATA_END, "4 last 24 hours: after window not complete")
    .otherwise("5 after the data window")
)

display(
    release_df
    .withColumn("window_position", window_position)
    .groupBy("window_position")
    .agg(
        F.count("*").alias("release_events"),
        F.min("release_published_at").alias("earliest_published_at"),
        F.max("release_published_at").alias("latest_published_at"),
    )
    .orderBy("window_position")
)

In [0]:
# R8.3 Preview of the seven anchor rules

# Rules 1 to 5 look at one release event at a time.
# The result is NULL when the event passes all five.
rules_1_to_5 = (
    F.when(
        F.col("release_action").isNull() | (F.col("release_action") != "published"),
        "1 NOT_PUBLISHED",
    )
    .when(published_at.isNull(), "2 MISSING_PUBLISHED_AT")
    .when(
        F.col("release_draft").isNull() | F.col("release_draft"),
        "3 DRAFT_OR_UNKNOWN",
    )
    .when(F.col("release_prerelease").isNull(), "4 PRERELEASE_UNKNOWN")
    .when(F.col("release_id").isNull(), "5 MISSING_RELEASE_ID")
)

# Rule 6 numbers the events of one release from the earliest to the latest.
# Events that passed rules 1 to 5 are numbered separately from the others,
# so an event that already failed never takes position 1.
earliest_first = (
    Window
    .partitionBy("repo_id", "release_id", "passed_rules_1_to_5")
    .orderBy("event_timestamp", "event_id")
)

anchor_preview_df = (
    release_df
    .withColumn("failed_rule", rules_1_to_5)
    .withColumn("passed_rules_1_to_5", F.col("failed_rule").isNull())
    .withColumn("position", F.row_number().over(earliest_first))
    .withColumn(
        "result",
        F.when(F.col("failed_rule").isNotNull(), F.col("failed_rule"))
        .when(F.col("position") > 1, "6 DUPLICATE_RELEASE_EVENT")
        .when(window_position != "3 both windows fit", "7 WINDOW_NOT_COVERED")
        .otherwise("ANCHOR"),
    )
)

display(
    anchor_preview_df
    .groupBy("result")
    .agg(
        F.count("*").alias("release_events"),
        count_when(F.col("release_prerelease")).alias("of_those_prereleases"),
    )
    .orderBy("result")
)

## 9. Releases per repository and the activity around them (G5)

To build G5, every release is joined with the activity events of its repository (push, pull request, issue, watch and fork events), and only the events inside the 48 hours around the release are kept. A repository with many releases **and** a lot of activity produces many pairs in that join. This check measures how large the join can get before the time filter.

`pairs_before_time_filter` = release events × activity events of the same repository. It is an upper limit. The real join is smaller, because of the anchor rules and the time filter.

- **R9.1** How many repositories have 1, 2 to 5, 6 to 20, 21 to 100 or more than 100 release events.
- **R9.2** The total size: all repositories with releases together, and the largest single repository.
- **R9.3** The 20 repositories with the most release events, with their activity.
- **R9.4** The 20 repositories with the most pairs. These decide whether the join needs special care.

Expected: 62,125 repositories with at least one release event.

In [0]:
# R9.1 Release events per repository

# One row per repository that has release events.
releases_per_repo_df = (
    release_df
    .groupBy("repo_id")
    .agg(
        F.count("*").alias("release_events"),
        F.countDistinct("release_id").alias("releases"),
        F.max("repo_name").alias("repo_name"),
    )
)

display(
    releases_per_repo_df
    .withColumn(
        "release_events_in_week",
        F.when(F.col("release_events") == 1, "1: one")
        .when(F.col("release_events") <= 5, "2: 2 to 5")
        .when(F.col("release_events") <= 20, "3: 6 to 20")
        .when(F.col("release_events") <= 100, "4: 21 to 100")
        .otherwise("5: more than 100"),
    )
    .groupBy("release_events_in_week")
    .agg(
        F.count("*").alias("repositories"),
        F.sum("release_events").alias("release_events"),
    )
    .orderBy("release_events_in_week")
)

In [0]:
# R9.2 Total size of the G5 join before the time filter

# Activity events per repository, from the five activity tables of G5.
activity_per_repo_df = (
    stack(G5_ACTIVITY_SOURCES)
    .groupBy("repo_id")
    .agg(F.count("*").alias("activity_events"))
)

# Keep every repository with releases, also when it has no activity at all.
join_size_df = (
    releases_per_repo_df
    .join(activity_per_repo_df, "repo_id", "left")
    .withColumn(
        "activity_events",
        F.coalesce(F.col("activity_events"), F.lit(0)),
    )
    .withColumn(
        "pairs_before_time_filter",
        F.col("release_events") * F.col("activity_events"),
    )
)

display(
    join_size_df
    .agg(
        F.count("*").alias("repositories_with_releases"),
        F.sum("release_events").alias("release_events"),
        F.sum("activity_events").alias("activity_events_in_those_repositories"),
        count_when(F.col("activity_events") == 0).alias("repositories_without_activity"),
        F.sum("pairs_before_time_filter").alias("pairs_before_time_filter"),
        F.max("pairs_before_time_filter").alias("largest_single_repository_pairs"),
    )
)

In [0]:
# R9.3 The 20 repositories with the most release events

display(
    join_size_df
    .select(
        "repo_name",
        "repo_id",
        "release_events",
        "releases",
        "activity_events",
        "pairs_before_time_filter",
    )
    .orderBy(F.desc("release_events"), "repo_id")
    .limit(20)
)

In [0]:
# R9.4 The 20 repositories with the most pairs

display(
    join_size_df
    .select(
        "repo_name",
        "repo_id",
        "release_events",
        "releases",
        "activity_events",
        "pairs_before_time_filter",
    )
    .orderBy(F.desc("pairs_before_time_filter"), "repo_id")
    .limit(20)
)

## 10. Repositories with more than one name

Gold groups and joins by `repo_id`. `repo_name` is only a label for the reader, because a repository can be renamed or moved to another owner. The label rule is: the latest name seen that day. This check shows how often the name really changes.

- **R10.1** Repositories by the number of different names in the week. Expected: almost all have one name.
- **R10.2** Repository-days with more than one name on the same day. Only on those days does "the latest name that day" have to choose.
- **R10.3** The opposite case: names that belong to more than one `repo_id`. This happens when a repository is deleted and created again under the same name. It is the reason a name is never used as a key.
- **R10.4** A sample of 20 repositories with more than one name.

In [0]:
# R10.1 Repositories by number of different names in the week

all_events_df = stack(G6_SOURCES)

# Step 1: one row per repository and name.
repo_names_df = (
    all_events_df
    .groupBy("repo_id", "repo_name")
    .agg(F.count("*").alias("events"))
)

# Step 2: one row per repository.
# count("repo_name") counts the names that are not NULL.
names_per_repo_df = (
    repo_names_df
    .groupBy("repo_id")
    .agg(
        F.count("repo_name").alias("names_in_week"),
        F.sum("events").alias("events"),
        F.concat_ws(
            " | ",
            F.sort_array(F.collect_list("repo_name")),
        ).alias("names"),
    )
)

display(
    names_per_repo_df
    .withColumn(
        "different_names",
        F.when(F.col("names_in_week") == 0, "0 names (name always missing)")
        .when(F.col("names_in_week") == 1, "1 name")
        .when(F.col("names_in_week") == 2, "2 names")
        .otherwise("3 or more names"),
    )
    .groupBy("different_names")
    .agg(
        F.count("*").alias("repositories"),
        F.sum("events").alias("events"),
    )
    .orderBy("different_names")
)

In [0]:
# R10.2 Repository-days with more than one name on the same day

display(
    all_events_df
    .groupBy("repo_id", "event_date")
    .agg(F.countDistinct("repo_name").alias("names_that_day"))
    .agg(
        F.count("*").alias("repository_days"),
        count_when(F.col("names_that_day") > 1).alias("days_with_more_than_one_name"),
        count_when(F.col("names_that_day") == 0).alias("days_without_a_name"),
    )
)

In [0]:
# R10.3 Names that belong to more than one repo_id

# repo_names_df has one row per repository and name,
# so counting its rows per name counts the repositories.
display(
    repo_names_df
    .filter(F.col("repo_name").isNotNull())
    .groupBy("repo_name")
    .agg(F.count("*").alias("repo_ids"))
    .agg(
        F.count("*").alias("different_names"),
        count_when(F.col("repo_ids") > 1).alias("names_with_more_than_one_repo_id"),
        F.max("repo_ids").alias("most_repo_ids_for_one_name"),
    )
)

In [0]:
# R10.4 A sample of 20 repositories with more than one name

display(
    names_per_repo_df
    .filter(F.col("names_in_week") > 1)
    .select("repo_id", "names_in_week", "events", "names")
    .orderBy(F.desc("events"), "repo_id")
    .limit(20)
)

## 11. Bots and the most active actors (G3)

G3 measures how the engineering work of a repository is spread over people. It reads six Silver tables: push, pull request, issue, issue comment, review and inline review comment events. Watch, fork and release events are not part of it.

GitHub Apps have a login that ends in `[bot]`, for example `dependabot[bot]`. Version 1 of G3 keeps every actor, including bots, and says so in the documentation. This check shows how much that matters. A login is only a label, so the rule "ends in `[bot]`" misses automation accounts with a normal login.

- **R11.1** Events from logins that end in `[bot]`, per Silver table and for all six together.
- **R11.2** Actors and events per login type.
- **R11.3** The 20 most active actors with their share of all engineering events.

In [0]:
# R11.1 Events from logins that end in [bot], per Silver table

engineering_df = (
    stack(G3_SOURCES)
    .withColumn(
        "login_type",
        F.when(F.col("actor_login").isNull(), "login missing")
        .when(F.col("actor_login").endswith("[bot]"), "ends in [bot]")
        .otherwise("other login"),
    )
)

is_bot_login = F.col("login_type") == "ends in [bot]"

bot_share_parts = [
    # One row per Silver table.
    engineering_df
    .groupBy("silver_table")
    .agg(
        F.count("*").alias("events"),
        count_when(is_bot_login).alias("bot_login_events"),
    ),
    # One row for all six tables together.
    engineering_df
    .agg(
        F.count("*").alias("events"),
        count_when(is_bot_login).alias("bot_login_events"),
    )
    .select(F.lit("ALL SIX TABLES").alias("silver_table"), "*"),
]

display(
    union_all(bot_share_parts)
    .withColumn(
        "bot_login_pct",
        F.round(
            F.try_divide(100 * F.col("bot_login_events"), F.col("events")),
            2,
        ),
    )
    .orderBy(F.desc("events"))
)

In [0]:
# R11.2 Actors and events per login type

display(
    engineering_df
    .groupBy("login_type")
    .agg(
        F.countDistinct("actor_id").alias("actors"),
        F.count("*").alias("events"),
        count_when(F.col("actor_id").isNull()).alias("events_without_actor_id"),
    )
    .orderBy(F.desc("events"))
)

In [0]:
# R11.3 The 20 most active actors

engineering_events_total = engineering_df.count()

display(
    engineering_df
    .groupBy("actor_id")
    .agg(
        F.max("actor_login").alias("actor_login"),
        F.count("*").alias("events"),
        F.countDistinct("repo_id").alias("repositories"),
    )
    .withColumn(
        "pct_of_all_engineering_events",
        F.round(
            F.try_divide(100 * F.col("events"), F.lit(engineering_events_total)),
            2,
        ),
    )
    .orderBy(F.desc("events"), "actor_id")
    .limit(20)
)

## 12. Is `event_id` unique across all nine Silver tables?

Each Silver notebook removes duplicates inside its own table. G3 and G6 place several Silver tables under each other, so the same `event_id` must not appear in two tables. Otherwise one event would be counted twice.

- **R12.1** Rows and different event IDs in the nine tables together. Expected: both are 33,623,791 and `null_event_ids` is 0.
- **R12.2** Event IDs that appear more than once, per Silver table. Expected: no rows.
- **R12.3** A sample of up to 20 such rows. Expected: no rows.

In [0]:
# R12.1 Rows and different event IDs in the nine Silver tables together

# all_events_df is the nine Silver tables under each other (cell of R10.1).
all_event_ids_df = all_events_df.select("event_id", "silver_table")

display(
    all_event_ids_df
    .agg(
        F.count("*").alias("rows_in_nine_tables"),
        F.countDistinct("event_id").alias("different_event_ids"),
        count_when(F.col("event_id").isNull()).alias("null_event_ids"),
    )
)

In [0]:
# R12.2 Event IDs that appear more than once (expected: no rows)

repeated_ids_df = (
    all_event_ids_df
    .groupBy("event_id")
    .agg(F.count("*").alias("times_found"))
    .filter(F.col("times_found") > 1)
)

# All rows that carry one of the repeated IDs.
repeated_rows_df = all_event_ids_df.join(repeated_ids_df, "event_id")

display(
    repeated_rows_df
    .groupBy("silver_table")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("event_id").alias("repeated_event_ids"),
    )
    .orderBy("silver_table")
)

In [0]:
# R12.3 A sample of up to 20 rows with a repeated event ID (expected: no rows)

display(
    repeated_rows_df
    .orderBy("event_id", "silver_table")
    .limit(20)
)

## What happens next

1. Send the notebook back with its results.
2. We go through the result tables together and note every rule they change.
3. The rules are then frozen in `docs/gold_contracts.md` (step 3 of the plan). Only after that is the first Gold table built.

Nothing in this notebook wrote to a table.